### DETR model training ###
Build the model training script

In [1]:
import sys
import os
import json
import numpy as np
import pandas as pd
import datetime
import logging
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib import patches

# PyTorch
import torch
from torch.utils.data import DataLoader

# Hugging Face Library
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor
from transformers import TrainingArguments, Trainer

%load_ext autoreload
%autoreload 2
import computervision
from computervision.imageproc import is_image
from computervision.datasets import DETRdataset
from computervision.transformations import AugmentationTransform
from computervision.mapeval import MAPEvaluator
from computervision.inference import get_gpu_info

print(f'Project version: {computervision.__version__}')
print(f'Python version:  {sys.version}')

Project version: v0.1.0
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


### Model name and directory ###

In [2]:
# Training data
model_version = 1
date_str = datetime.date.today().strftime('%y%m%d')
model_name = f'rtdetr_dentex_{date_str}_{str(model_version).zfill(2)}'

data_dir = os.path.join(os.environ.get('DATA_DIR'), 'computervision_data')
dataset_dir = os.path.join(data_dir, 'dentex')

model_dir = os.path.join(dataset_dir, 'model')
model_name_dir = os.path.join(model_dir, model_name)
Path(model_name_dir).mkdir(parents=True, exist_ok=True)

print(f'Model name:   {model_name}')
print(f'data folder:  {data_dir}')
print(f'Model folder: {model_name_dir}')

Model name:   rtdetr_dentex_260222_01
data folder:  /app/data/computervision_data
Model folder: /app/data/computervision_data/dentex/model/rtdetr_dentex_260222_01


### Training and validation data ###

In [3]:
dataset_name = 'dataset_object_dentex'
train_image_dir = os.path.join(dataset_dir, dataset_name)
train_annotation_file_name = f'{dataset_name}_dset.parquet'
train_annotation_file = os.path.join(train_image_dir, train_annotation_file_name)

val_image_dir = os.path.join(train_image_dir,'test')
val_annotation_file_name = f'{dataset_name}_test.parquet'
val_annotation_file = os.path.join(val_image_dir, val_annotation_file_name)

# Column names for the annotation files
tooth_pos_col = 'ada'
file_name_col = 'file_name'
bbox_col = 'bbox'
dset_col = 'dset'

### Model training parameters ###

In [4]:
# Training and model parameters
device_number = 0
device, device_str = get_gpu_info(device_number=device_number)

# Image transformations for training and validation
im_width, im_height = 640, 640
# Augmentations
train_quadrants = [12, 34, 14, 23]
val_quadrants = [14, 23]
train_transform_name = 'train_dentex'
val_transform_name = 'val'
aug = AugmentationTransform(im_width=im_width, im_height=im_height)
train_transforms = aug.get_transforms(name=train_transform_name)
val_transforms = aug.get_transforms(name=val_transform_name)

# Important information about the model that we want to save
model_info = {'model_version': model_version,
              'device_number': device_number,
              'project_version': computervision.__version__,
              'model_name': model_name,
              'train_image_dir': train_image_dir,
              'val_image_dir': val_image_dir,
              'im_width': im_width,
              'im_height': im_height,
              'hf_checkpoint': 'PekingU/rtdetr_v2_r101vd',
              'training_checkpoint': 'PekingU/rtdetr_v2_r101vd',
              'train_quadrants': train_quadrants,
              'val_quadrants': val_quadrants,
              'train_transform_name': train_transform_name,
              'val_transform_name': val_transform_name,
              'val_score_threshold': 0.05}

# Specific arguments for the Trainer. 48
# See: https://huggingface.co/docs/transformers/en/main_classes/trainer#trainer
training_args = {'output_dir': model_name_dir,
                 'num_train_epochs': 2,
                 'dataloader_num_workers': 0,
                 'max_grad_norm': 0.1,
                 'learning_rate': 5e-5,
                 'warmup_steps': 300,
                 'per_device_train_batch_size': 4,
                 'dataloader_num_workers': 8,
                 'metric_for_best_model': 'eval_map',
                 'greater_is_better': True,
                 'load_best_model_at_end': True,
                 'eval_strategy': 'epoch',
                 'save_strategy': 'epoch',
                 'save_total_limit': 5,
                 'remove_unused_columns': False,
                 'eval_do_concat_batches': False}

# We want to maintain the aspect ratio of the images
# So, we resize the image first and then pad it
processor_params = {'do_resize': True,
                    'size': {'max_height': im_height,
                             'max_width': im_width},
                    'do_pad': True,
                    'pad_size': {'height': im_height,
                                 'width': im_width}}

# Bounding box format for the annotations
bbox_format = {'format': 'coco',
               'label_fields': ['tooth_position'],
               'clip': True,
               'filter_invalid_bboxes': True,
               'min_area': 10000}

Current device:    cuda:0


### Check training and validation data ###

In [5]:
train_df = pd.read_parquet(train_annotation_file)
train_df = train_df.loc[
    (train_df[dset_col] == 'train') &
    (train_df['quadrants'].isin(train_quadrants))].astype({'ada': int})

# Filter the validation images and quadrants and take only the first augmentation
val_df = pd.read_parquet(val_annotation_file)
val_df = val_df.loc[
    (val_df[dset_col] == 'val') &
    (val_df['quadrants'].isin(val_quadrants)) &
    (val_df['transformation'] == 0)].astype({'ada': int})

# Check the images on disk
train_file_list = list(train_df[file_name_col].unique())
train_checked = np.sum([is_image(os.path.join(train_image_dir, file)) for file in train_file_list])
print(f'Images in training data:         {len(train_file_list)}')
print(f'Files checked in training data:  {train_checked}')
print(f'Annotations in training data:    {train_df.shape[0]}')

print()

val_file_list = list(val_df[file_name_col].unique())
val_checked = np.sum([is_image(os.path.join(val_image_dir, file)) for file in val_file_list])
print(f'Images in validation data:       {len(val_file_list)}')
print(f'Files checked in val data:       {val_checked}')
print(f'Annotations in validation data:  {val_df.shape[0]}')

# Create the label ids (tooth position, but starting from 0)
# The model needs label ids, not labels. So we need to add a label id column
label_name_list = sorted(list(train_df[tooth_pos_col].unique()))
id2label = dict(zip(range(len(label_name_list)), label_name_list))
id2label = {int(label_id): str(label_name) for label_id, label_name in id2label.items()}
label2id = {str(label_name): int(label_id) for label_id, label_name in id2label.items()}

train_df = train_df.assign(label=train_df[tooth_pos_col].apply(lambda name: label2id.get(str(name))))
val_df = val_df.assign(label=val_df[tooth_pos_col].apply(lambda name: label2id.get(str(name))))

Images in training data:         2391
Files checked in training data:  2391
Annotations in training data:    34166

Images in validation data:       32
Files checked in val data:       32
Annotations in validation data:  389


### Save the model configuration and create a logger ###

In [6]:
parameters = {'model_info': model_info,
              'id2label': id2label,
              'training_args': training_args,
              'processor_params': processor_params,
              'bbox_format': bbox_format}

json_file = os.path.join(model_name_dir, f'{model_name}.json')
with open(json_file, 'w') as f:
    json.dump(parameters, f, indent=4) # indent for pretty-printing

# Set up the logger
log_file_name = f'{model_name}.log'
log_file = os.path.join(model_name_dir, log_file_name)
dtfmt = '%y%m%d-%H:%M'
logfmt = '%(asctime)s-%(name)s-%(levelname)s-%(message)s'

logging.basicConfig(filename=log_file,
                    filemode='w',
                    level=logging.INFO,
                    format=logfmt,
                    datefmt=dtfmt,
                    force=True)

logger = logging.getLogger(name=__name__)

### Initialize the model from a checkpoint ###
The warnings tell us that the number of classes in the data set don't match with the training data.
That's OK because we will re-train the model on the new data.

In [7]:
model_checkpoint = model_info.get('hf_checkpoint')
processor = RTDetrImageProcessor.\
    from_pretrained(model_checkpoint, **processor_params)

# Load model from a pretrained checkpoint
training_checkpoint = model_info.get('training_checkpoint')
model = RTDetrV2ForObjectDetection.\
    from_pretrained(training_checkpoint,
                    id2label=id2label,
                    label2id=label2id,
                    anchor_image_size=None,
                    ignore_mismatched_sizes=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1025 [00:00<?, ?it/s]

RTDetrV2ForObjectDetection LOAD REPORT from: PekingU/rtdetr_v2_r101vd
Key                                                 | Status   |                                                                                         
----------------------------------------------------+----------+-----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([32, 256])
model.denoising_class_embed.weight                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([33, 256])
model.enc_score_head.weight                         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([32, 256])
model.decoder.class_embed.{0, 1, 2, 3, 4, 5}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([32])          
model.

### PyTorch Datasets ###

In [8]:
train_dataset = DETRdataset(data=train_df,
                            image_processor=processor,
                            image_dir=train_image_dir,
                            file_name_col=file_name_col,
                            label_id_col='label',
                            bbox_col=bbox_col,
                            transforms=train_transforms)

val_dataset = DETRdataset(data=val_df,
                          image_processor=processor,
                          image_dir=val_image_dir,
                          file_name_col=file_name_col,
                          label_id_col='label',
                          bbox_col=bbox_col,
                          transforms=val_transforms)

### Set up the Trainer and start training ###

The RuntimeWarning: invalid value encountered in divide visibility_ratios = remaining_areas / box_areas in Albumentations occurs when an augmentation results in a bounding box having a zero area. This typically happens when cropping or other spatial transforms cause a bounding box to be completely cut out of the image. The division by zero is handled internally, but the warning signals that a box was dropped. 

In [10]:
def collate_fn(batch):
    """
    Collates a batch of data samples into a single dictionary for model input.
    """
    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in batch])
    data["labels"] = [x["labels"] for x in batch]
    return data

# Set the evaluation metrics
eval_compute_metrics_fn = MAPEvaluator(image_processor=processor,
                                       threshold=model_info.get('val_score_threshold'),
                                       id2label=id2label)

training_arguments = TrainingArguments(**training_args)

trainer = Trainer(model=model,
                  args=training_arguments,
                  train_dataset=train_dataset,
                  eval_dataset=val_dataset,
                  processing_class=processor,
                  data_collator=collate_fn,
                  compute_metrics=eval_compute_metrics_fn)

trainer.train()

Epoch,Training Loss,Validation Loss,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map 1,Mar 100 1,Map 2,Mar 100 2,Map 3,Mar 100 3,Map 4,Mar 100 4,Map 5,Mar 100 5,Map 6,Mar 100 6,Map 7,Mar 100 7,Map 8,Mar 100 8,Map 9,Mar 100 9,Map 10,Mar 100 10,Map 11,Mar 100 11,Map 12,Mar 100 12,Map 13,Mar 100 13,Map 14,Mar 100 14,Map 15,Mar 100 15,Map 16,Mar 100 16,Map 17,Mar 100 17,Map 18,Mar 100 18,Map 19,Mar 100 19,Map 20,Mar 100 20,Map 21,Mar 100 21,Map 22,Mar 100 22,Map 23,Mar 100 23,Map 24,Mar 100 24,Map 25,Mar 100 25,Map 26,Mar 100 26,Map 27,Mar 100 27,Map 28,Mar 100 28,Map 29,Mar 100 29,Map 30,Mar 100 30,Map 31,Mar 100 31,Map 32,Mar 100 32
1,32.801988,11.426087,0.459100,0.737600,0.497200,-1.000000,0.308300,0.478000,0.511100,0.680700,0.687100,-1.000000,0.488900,0.695300,0.434400,0.644400,0.383900,0.730800,0.303900,0.680000,0.156500,0.723100,0.301200,0.726700,0.620200,0.780000,0.521800,0.653800,0.356500,0.766700,0.531200,0.714300,0.528200,0.646200,0.536700,0.657100,0.319300,0.650000,0.168900,0.638500,0.381700,0.630800,0.388500,0.707100,0.389400,0.700000,0.520300,0.655600,0.599800,0.728600,0.515100,0.663600,0.541300,0.653300,0.468400,0.671400,0.650300,0.731200,0.532600,0.691700,0.475100,0.742900,0.198300,0.533300,0.500800,0.633300,0.699300,0.766700,0.519900,0.713300,0.523700,0.646200,0.601400,0.740000,0.511800,0.720000,0.511100,0.645500
2,21.375010,9.444963,0.614100,0.898500,0.718000,-1.000000,0.388200,0.628200,0.684900,0.732900,0.732900,-1.000000,0.550000,0.739400,0.545000,0.711100,0.626600,0.776900,0.555700,0.706700,0.441700,0.684600,0.662600,0.726700,0.694500,0.760000,0.613500,0.692300,0.650600,0.777800,0.679400,0.742900,0.580000,0.692300,0.680400,0.721400,0.539800,0.657100,0.340600,0.684600,0.663700,0.707700,0.660700,0.757100,0.500100,0.711100,0.620200,0.722200,0.751100,0.828600,0.741200,0.772700,0.698600,0.740000,0.713600,0.800000,0.772700,0.837500,0.572100,0.725000,0.497800,0.685700,0.217900,0.550000,0.658200,0.716700,0.763500,0.833300,0.709100,0.780000,0.674500,0.715400,0.736800,0.800000,0.647700,0.753300,0.440700,0.681800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.aifi.0.layers.0.self_attn.k_proj.weight', 'model.encoder.aifi.0.layers.0.self_attn.k_proj.bias', 'model.encoder.aifi.0.layers.0.self_attn.v_proj.weight', 'model.encoder.aifi.0.layers.0.self_attn.v_proj.bias', 'model.encoder.aifi.0.layers.0.self_attn.q_proj.weight', 'model.encoder.aifi.0.layers.0.self_attn.q_proj.bias', 'model.encoder.aifi.0.layers.0.self_attn.o_proj.weight', 'model.encoder.aifi.0.layers.0.self_attn.o_proj.bias', 'model.encoder.aifi.0.layers.0.self_attn_layer_norm.weight', 'model.encoder.aifi.0.layers.0.self_attn_layer_norm.bias', 'model.encoder.aifi.0.layers.0.mlp.fc1.weight', 'model.encoder.aifi.0.layers.0.mlp.fc1.bias', 'model.encoder.aifi.0.layers.0.mlp.fc2.weight', 'model.encoder.aifi.0.layers.0.mlp.fc2.bias', 'model.encoder.aifi.0.layers.0.final_layer_norm.weight', 'model.encoder.aifi.0.layers.0.final_layer_norm.bias', 'model.decoder.layers.0.self_attn.o_proj.weight', 'model.decoder.layers.0.s

TrainOutput(global_step=1196, training_loss=25.91080518231344, metrics={'train_runtime': 199.6966, 'train_samples_per_second': 23.946, 'train_steps_per_second': 5.989, 'total_flos': 2.695348067401728e+18, 'train_loss': 25.91080518231344, 'epoch': 2.0})